In [1]:
!python -m venv .venv

In [2]:
!source .venv/bin/activate

In [3]:
!pip install torch torchvision pillow numpy scikit-learn tqdm certifi opencv-python ultralytics google-genai python-dotenv ultralytics

In [4]:
import os
import shutil
from pathlib import Path
import certifi
import torch
from ultralytics import YOLO

os.environ.setdefault("SSL_CERT_FILE", certifi.where())

'/opt/anaconda3/lib/python3.14/site-packages/certifi/cacert.pem'

In [5]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [6]:
MODEL_CHOICES = [
    "yolo26n.pt", "yolo26s.pt", "yolo26m.pt", "yolo26l.pt", "yolo26x.pt",
]

In [7]:
data = "data_detect/data.yaml"
model_name = "yolo26n.pt"
epochs = 13
batch_size = -1
img_size = 640
scale = 0.9
perspective = 0.0005  # random perspective warp strength (Ultralytics 'perspective' aug)
output = None  # Output path. Defaults to models/<model_name>_trained.pt

In [8]:
if output is None:
    model_basename = model_name.replace(".pt", "")
    output = f"models/{model_basename}_trained.pt"

if not Path(data).exists():
    raise FileNotFoundError(
        f"No dataset at '{data}'. Run src/annotate_bboxes.py then "
        f"src/prepare_detect_dataset.py first."
    )

device = str(get_device())
print(f"Using device: {device}")

Using device: mps


In [9]:
model = YOLO(model_name)
train_results = model.train(
    data=str(Path(data).resolve()),
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    device=device,
    scale=scale,
    perspective=perspective,
    name="box_open_closed_yolo_detect",
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.124 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.123 🚀 Python-3.14.6 torch-2.13.0 MPS (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/nagendra.rishab.in/Desktop/SelfProjects/modelretrain/data_detect/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=13, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, 

/opt/anaconda3/lib/python3.14/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/13      5.29G      1.058      3.098   0.009523         42        640: 100% ━━━━━━━━━━━━ 859/859 1.7s/it 24:032.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.9s/it 3.7s9.6s
                   all         35         23       0.46      0.357      0.466      0.317

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size


/opt/anaconda3/lib/python3.14/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/13      5.28G      1.005     0.8181   0.009428         35        640: 100% ━━━━━━━━━━━━ 859/859 1.7s/it 24:071.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1s/it 2.2s6.2s
                   all         35         23      0.521      0.952      0.595      0.407

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size


/opt/anaconda3/lib/python3.14/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       3/13      5.28G     0.9784     0.6407   0.009103         44        640: 72% ━━━━━━━━╸─── 619/859 1.4s/it 17:01<5:473


KeyboardInterrupt: 

In [ ]:
save_dir = Path(getattr(train_results, "save_dir", None) or model.trainer.save_dir)
best_ckpt = save_dir / "weights" / "best.pt"

Path(output).parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_ckpt, output)
print(f"Saved best model to {output} (full run artifacts in {save_dir})")

best_model = YOLO(str(best_ckpt))
metrics = best_model.val(data=str(Path(data).resolve()), split="test", imgsz=img_size, device=device)
print(f"\nTest mAP50: {metrics.box.map50:.4f}  mAP50-95: {metrics.box.map:.4f}")
print(f"Full report saved under: {metrics.save_dir}")